In [0]:
# =================================================================================
# ewm_continuous_ingestion.py
#
# A generic, parameter-driven continuous streaming job (Job B).
# Reads source data, checks for schema changes, triggers backfills,
# and writes to a target Delta table.
# =================================================================================

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 1: Get Parameters and Configure Paths
# MAGIC Fetch all configuration from Databricks job widgets to make this notebook fully generic.

# COMMAND ----------

# --- 1. Fetch all parameters from job widgets ---
target_table_name = dbutils.widgets.get("target_table_name")
source_data_path = dbutils.widgets.get("source_data_path")
control_table_schema = dbutils.widgets.get("control_table_schema")
checkpoint_location_root = dbutils.widgets.get("checkpoint_location_root")
ADLS_STORAGE_ACCOUNT_NAME = dbutils.widgets.get("storage_account_name")
adls_secret_scope = dbutils.widgets.get("secret_scope_name")
adls_secret_key = dbutils.widgets.get("secret_key_name")

print(f"Received Job Parameters:")
print(f"  - target_table_name:      {target_table_name}")
print(f"  - source_data_path:       {source_data_path}")
print(f"  - control_table_schema:   {control_table_schema}")
print(f"  - checkpoint_location_root: {checkpoint_location_root}")
print(f"  - storage_account_name:   {ADLS_STORAGE_ACCOUNT_NAME}")
print(f"  - adls_secret_scope:      {adls_secret_scope}")
print(f"  - adls_secret_key:        {adls_secret_key}")

# --- 2. Define all table and path names based on parameters ---
catalog, target_db, target_tbl = target_table_name.split('.')
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"
checkpoint_path = f"{checkpoint_location_root.rstrip('/')}/{target_tbl}_checkpoint"

# Enable auto-merge for schema evolution
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# COMMAND ----------

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 2: Get Additional Parameters & Configure Authentication

# COMMAND ----------

import requests
import json
from pyspark.sql.functions import col, from_json, lit
from pyspark.sql.types import StructType

# === NEW: Get the Backfill Job ID directly as a parameter ===
# This is more robust than searching by name.
backfill_job_id = int(dbutils.widgets.get("backfill_job_id"))
print(f"Received Backfill Job ID: {backfill_job_id}")

# === 1. ADLS Storage Authentication (remains the same) ===
spark.conf.set(
    f"fs.azure.account.key.{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    dbutils.secrets.get(scope=adls_secret_scope, key=adls_secret_key)
)
print("✅ ADLS storage access configured.")

# === 2. Databricks API Configuration (remains the same) ===
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")
API_SECRET_SCOPE = "my_jobb_scope"
API_SECRET_KEY = "job-b-api-token"
API_TOKEN = dbutils.secrets.get(scope=API_SECRET_SCOPE, key=API_SECRET_KEY)
HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}
print("✅ Databricks API configured for backfill trigger.")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 3: Utility Functions (Updated)

# COMMAND ----------

def _trigger_backfill_job(job_id, old_hash, new_hash, table_name, control_schema):
    """Triggers the backfill job using its direct ID."""
    print(f"Attempting to trigger backfill for Job ID: {job_id}")
    params = {
        "notebook_params": {
            "target_table_name": table_name,
            "old_schema_hash": old_hash,
            "new_schema_hash": new_hash,
            "run_mode": "WET_RUN",
            "control_table_schema": control_schema
        }
    }
    response = requests.post(
        f"https://{DATABRICKS_HOST}/api/2.1/jobs/run-now",
        headers=HEADERS,
        json={"job_id": job_id, **params}
    )
    if response.status_code == 200:
        run_id = response.json()["run_id"]
        print(f"✅ Successfully triggered backfill job. Run ID: {run_id}")
        return run_id
    else:
        error_msg = response.text
        # This will now provide a much more detailed error if the run fails.
        raise RuntimeError(f"❌ Failed to trigger backfill job ID '{job_id}'. Status: {response.status_code}. API Error: {error_msg}")

def get_latest_schema_info(transition_table, target_tbl_name):
    """Gets the latest schema hash, the previous schema hash, and the backfill status."""
    print(f"Querying for schema info from '{transition_table}' for table '{target_tbl_name}'...")
    history_df = spark.sql(f"""
        SELECT schema_hash, backfill_status
        FROM {transition_table}
        WHERE target_table_name = '{target_tbl_name}'
        ORDER BY event_ts DESC
        LIMIT 2
    """)
    results = history_df.collect()

    if not results:
        return None, None, None

    latest_schema_hash = results[0]['schema_hash']
    backfill_status = results[0]['backfill_status']
    old_schema_hash = results[1]['schema_hash'] if len(results) > 1 else None
    
    print(f"Schema Info Found: latest_hash='{latest_schema_hash}', old_hash='{old_schema_hash}', status='{backfill_status}'")
    return latest_schema_hash, backfill_status, old_schema_hash

def get_schema_for_hash(columns_table, schema_hash):
    """Reconstructs a Spark schema from the schema_store_columns table."""
    rows = spark.sql(f"SELECT name, type_json FROM {columns_table} WHERE schema_hash = '{schema_hash}'").collect()
    fields = [StructField(row['name'], StructType.fromJson(json.loads(row['type_json']))) for row in rows]
    return StructType(fields)


# COMMAND ----------

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 4: Main Streaming Logic (`foreachBatch`)

# COMMAND ----------

def process_microbatch(batch_df, batch_id):
    """The main logic that runs for each microbatch of the stream."""
    print(f"\n--- Starting Microbatch ID: {batch_id} ---")
    
    # 1. Check for schema change.
    # This function now correctly finds the latest and previous schema hashes.
    latest_schema_hash, backfill_status, old_schema_hash = get_latest_schema_info(schema_transition_table, target_table_name)
    
    if not latest_schema_hash:
        raise RuntimeError(f"❌ No schema found in '{schema_transition_table}' for table '{target_table_name}'. Job A must run first.")

    # 2. If a backfill is pending, trigger it and then STOP the stream.
    if backfill_status == 'PENDING':
        print(f"📢 Backfill 'PENDING' for schema {latest_schema_hash}. Triggering backfill job...")
        
        # --- THIS IS THE FIX ---
        # The incorrect _get_job_id_by_name line is removed.
        # We now directly use the `backfill_job_id` variable passed in as a job parameter.
        _trigger_backfill_job(backfill_job_id, old_schema_hash, latest_schema_hash, target_table_name, control_table_schema)
        
        # This is the line that stops the stream. We'll address this when we implement the new design.
        print("🛑 Backfill triggered. Stopping the stream to allow for restart with the new schema.")
        spark.streams.active[0].stop()
        return

    # 3. If status is not PENDING, proceed to process the data for this microbatch.
    # Reconstruct the schema for the current "good" schema hash.
    print(f"🔒 Locking microbatch to schema hash: {latest_schema_hash} (Status: {backfill_status})")
    target_schema = get_schema_for_hash(schema_store_columns_table, latest_schema_hash)
    if not target_schema.fields:
        raise ValueError(f"Could not reconstruct schema for hash {latest_schema_hash} from {schema_store_columns_table}, or schema is empty.")
        
    # 4. Parse and flatten the data according to the target schema.
    # Assumes the source data is a single JSON string in a 'value' column.
    parsed_df = batch_df.withColumn("jsonData", from_json(col("value"), target_schema)).select("jsonData.*")
    
    # Add audit columns
    final_df = parsed_df.withColumn("ingest_batch_id", lit(batch_id)) \
                        .withColumn("loaded_timestamp", lit(spark.sql("SELECT current_timestamp()").first()[0])) \
                        .withColumn("schema_hash_at_ingest", lit(latest_schema_hash))

    # 5. Write data to the target Delta table.
    print(f"✍️ Writing {final_df.count()} records to {target_table_name}...")
    final_df.write.format("delta").mode("append").saveAsTable(target_table_name)
    print(f"✅ Microbatch ID: {batch_id} completed successfully.")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 5: Start the Stream

# COMMAND ----------

print("🚀 Starting the stream...")
print(f"Reading from source: {source_data_path}")
print(f"Using checkpoint: {checkpoint_path}")

# Assumes source data is JSON, with each file containing one JSON object per line.
# The raw JSON string is loaded into a column named "value".
raw_stream_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "parquet")  # 
         .option("cloudFiles.schemaLocation", checkpoint_path)
         .load(source_data_path) # <-- CORRECTED: Using parameterized path
)

# Start the streaming query
stream_writer = (
    raw_stream_df.writeStream
                 .outputMode("append")
                 .option("checkpointLocation", checkpoint_path)
                 .trigger(processingTime='30 seconds') # Continuous
                 .foreachBatch(process_microbatch)
                 .start()
)

stream_writer.awaitTermination()